# M1 v2 — direct prize-to-book support, same Qwen3-1.7B

This is a new prospective training comparison from the same pinned public M0 with fresh LoRA, not a continuation of the failed1200-step adapter. Original candidate v1.1 facts/splits, all5040 original seven-task targets, LoRA/LR/loss/precision and840-row development checks/gates stay fixed. Add720 auxiliary rows: all120 F/R units×three training-only prize-to-book questions×both formats. Total5760 rows,2880 per mode. No N or final-test training/evaluation.

Budget:1372 attempted/required updates =ceil(1200×5760/5040),21952 nominal examples,3.8111 dataset passes versus3.8095 before (relative difference0.0417%). This nearly matches row exposure; tokens/compute/update trajectories differ, so it is not a pure causal or compute-matched test. Source failed model/data/results remain preserved. Auxiliary supervision is a hypothesis, not a guaranteed fix.

Save/private-upload/hash-round-trip at343/686/1029/1372 attempts. Endpoint1372 only; no intermediate-dev selection, silent extension, precision/batch/mode fallback or automatic resume/retry. Checkpoints preserve artifacts but do not implement exact mid-run loader/RNG resume. First run a disposable memory probe in its own process; actual training starts fresh after source-matched fit and32 exact M0 fact/10 utility controls. Reuse the existing1680 M0 dev records with provenance. Same3422 planned new control/endpoint/reload outputs and original28 per-task plus family/utility/generation/update/reload/persistence gates; auxiliary task is training-only and adds no easier acceptance metric.

Use existing Kaggle T4/HF_TOKEN. Publish new runner/config/bank/tests/protocol/preflight first, then run all cells sequentially. All service/model work is in fresh subprocesses. Complete final model/report backup and the exact-ZIP private round-trip, then download the printed ZIP. This notebook has not run on GPU; no improved M1 result is claimed. Passing candidates still require independent review before canonical M1/M2.


In [ ]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle T4 kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='inverse-v2-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/training/inverse_m1.py').is_file(), 'Publish the prepared inverse-support v2 source/config/bank before running.'
config_path = 'configs/training/m1_full_qwen17_inverse_v2.json'


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))'], check=True)


In [ ]:
import json
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
config = json.loads(Path(config_path).read_text())
private_env = os.environ.copy()
private_env['HF_TOKEN'] = hf_token
try:
    subprocess.run([sys.executable, '-c', "import os; from huggingface_hub import HfApi; assert HfApi(token=os.environ['HF_TOKEN']).repo_info('tharundevulapalli/persistent-llm-unlearning-checkpoints', repo_type='model').private; print('Private artifact repository accessible; credential is not displayed.')"], env=private_env, check=True)
finally:
    private_env.pop('HF_TOKEN', None)
def run_inverse(mode, directory, fit_directory=None):
    child_env = os.environ.copy()
    child_env['HF_TOKEN'] = hf_token
    command = [sys.executable, '-u', '-m', 'src.training.inverse_m1', '--config', config_path, '--output-dir', str(directory), '--mode', mode]
    if fit_directory is not None:
        command += ['--fit-dir', str(fit_directory)]
    try:
        return subprocess.run(command, env=child_env, check=False)
    finally:
        child_env.pop('HF_TOKEN', None)


In [ ]:
prep_dir = Path(tempfile.mkdtemp(prefix='inverse-v2-prepare-', dir=remote_root))
assert run_inverse('prepare', prep_dir).returncode == 0
plan = json.loads((prep_dir / 'plan.json').read_text())
masking = json.loads((prep_dir / 'masking_report.json').read_text())
assert plan['training_rows'] == 5760 and plan['original_training_rows'] == 5040 and plan['auxiliary_rows'] == 720
assert plan['development_rows'] == 840 and plan['planned_update_attempts'] == 1372
assert plan['comparison']['old_targets_preserved_as_prefix'] and plan['comparison']['evaluation_inputs_and_gates_unchanged']
assert abs(plan['comparison']['relative_row_exposure_difference']) < 0.0005
assert masking['prefixes_verified'] and masking['truncated_examples'] == 0
assert all(value['rows'] == 2880 and value['max_tokens'] <= 256 for value in masking['by_mode'].values())
print(json.dumps(masking, indent=2))
print('Comparison/exposure policy:', json.dumps(plan['comparison'], indent=2))
print('New auxiliary coverage:', plan['augmentation']['inverse_support']['rows_by_partition'])


In [ ]:
from src.training.evidence_zip import archive_evidence

fit_dir = Path(tempfile.mkdtemp(prefix='inverse-v2-fit-', dir=remote_root))
fit_result = run_inverse('fit', fit_dir)
fit_archive = archive_evidence(fit_dir)
print('Disposable fit evidence:', fit_archive)
assert (fit_dir / 'fit_report.json').is_file(), 'Fit failed before a report; preserve the ZIP and inspect the error.'
fit_report = json.loads((fit_dir / 'fit_report.json').read_text())
print(json.dumps(fit_report, indent=2))
assert fit_result.returncode == 0 and fit_report['success'], 'Stop: fit did not pass. No automatic fallback or training retry.'
assert fit_report['temporary_optimizer_updates'] == 1 and not fit_report['checkpoint_saved']
print('Probe exited; its adapter is discarded. Actual training starts fresh in the next cell.')


In [ ]:
# ACTUAL v2 TRAINING: fresh public1.7B/new LoRA,1372 attempts, no old-M1 restoration.
assert fit_report['success']
output_dir = Path(tempfile.mkdtemp(prefix='m1-inverse-v2-', dir=remote_root))
print('Training outputs:', output_dir, flush=True)
print('Private checkpoint attempts:', config['checkpoint_attempts'], flush=True)
training_result = run_inverse('train', output_dir, fit_dir)
print('Training return code:', training_result.returncode)
if training_result.returncode == 2:
    print('Source M0 controls/utility failed before LoRA creation; preserve evidence, zero real updates.')
elif training_result.returncode != 0:
    print('Incomplete attempt: retain partial checkpoints/receipts and continue to backup/export cells.')


In [ ]:
summary_path = output_dir / 'summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print('Execution complete:', summary['execution_complete'], '| applied updates:', summary.get('optimizer_updates', 0))
    if 'm1_candidate' in summary:
        print('Candidate gate pass:', summary['m1_candidate']['passed'])
        print('Failed checks:', [key for key, value in summary['m1_candidate']['checks'].items() if not value])
        for stage, groups in summary['evaluation'].items():
            for name, result in groups.items():
                print(stage, name, '| answers:', str(result['answer_matches']) + '/' + str(result['count']), '| format:', result['format_violations'], '| caps:', result['token_limit_hits'])
        print('Thinking F prize-to-author:', summary['m1_candidate']['per_task']['thinking/F/prize_to_author'])
    else:
        print('Stop reason:', summary.get('stop_reason'))
else:
    print('No completed summary. Preserve partial logs/checkpoint receipts.')
comparison_path = output_dir / 'comparison_v1.json'
if comparison_path.is_file():
    comparison = json.loads(comparison_path.read_text())
    print('Original v1 failed gate remains:', comparison['control_verdict']['passed'])
    print('V2 endpoint result:', comparison['candidate_verdict']['passed'])
print('No canonical promotion until independent raw/private-backup review; no unlearning occurs here.')


In [ ]:
# Endpoint/optimizer/tokenizer/report backup, or clearly labelled incomplete evidence.
backup_result = run_inverse('backup', output_dir)
print('Model/report backup return code:', backup_result.returncode)
if backup_result.returncode != 0:
    print('Private model/report backup failed. Retain this runtime; preserve/download evidence with the next cell.')


In [ ]:
# Export text plus checkpoint JSON metadata and back up the ZIP itself privately.
export_result = run_inverse('export', output_dir)
archive = Path(str(output_dir) + '-evidence.zip')
if archive.is_file():
    print('DOWNLOAD THIS ZIP:', archive)
    print('ZIP receipt:', Path(str(archive) + '-persistence.json'))
if export_result.returncode != 0:
    print('Private ZIP backup failed or export stopped. Download any printed ZIP and retain the runtime/evidence for review.')
print('Model/optimizer binaries stay private; original v1 checkpoint/results/gates are preserved.')
